In [1]:
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
%%writefile requirements.txt

nltk>=3.0.0
notebook>=7.0.0
seaborn
torch>=2.0.0
torchvision
tqdm>=4.0.0

Writing requirements.txt


In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.1/915.1 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: jupyter-server
    Found existing installation: jupyter_server 2.12.5
    Uninstalling jupyter_server-2.12.5:
      Successfully uninstalled jupyter_server-2.12.5
  Attempting uninstall: jupyter-lsp
    Found existing installation: jupyter-lsp 1.5.1
    Uninstalling jupyter-lsp-1.5.1:
      Successfully uninstalled jupyter-lsp-1.5.1
  Attempting uninstall: jupyterlab
    Found existing installation: jupyterlab 3.6.8
    Uninstalling jupyterlab-3.6.8:
      Successfully uninstalled jupyterlab-3.6.8
  Attempting uninstall: notebook
    Found existing installation: notebook 6.

In [4]:
%%writefile utils.py

from torch import Tensor, optim


class AverageMeter:
    """
    跟踪指标的最近值、平均值、总和和计数
    """

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def adjust_learning_rate(optimizer: optim.Optimizer, shrink_factor: float):
    """
    按指定因子缩小学习率

    :param optimizer: 学习率需要缩小的优化器
    :param shrink_factor: 乘以学习率的因子，在区间(0, 1)内
    """
    for param_group in optimizer.param_groups:
        param_group["lr"] *= shrink_factor


def accuracy(scores: Tensor, targets: Tensor, k: int) -> float:
    """
    计算top-k准确率

    :param scores: 模型的输出分数
    :param targets: 真实标签
    :param k: top-k准确率中的k
    :return: top-k准确率
    """

    batch_size = targets.size(0)
    _, ind = scores.topk(k, 1, True, True)
    correct = ind.eq(targets.view(-1, 1).expand_as(ind))
    correct_total = correct.view(-1).float().sum()  # 0D张量
    return correct_total.item() * (100.0 / batch_size)

Writing utils.py


In [5]:
%%writefile caption_dataset.py

import json
import os
import typing
from pathlib import Path
from typing import Literal

import torch
from PIL import Image
from torch import Tensor
from torch.utils.data import Dataset
from tqdm import tqdm

PAD_TOKEN = 0
DatasetSplit = Literal["train", "val", "test"]


def collate_fn(batch: list[tuple]):
    """
    Collate function for CaptionDataset.
    """
    is_train = len(batch[0]) == 2

    if is_train:
        imgs, captions = zip(*batch)
    else:
        imgs, captions, all_captions = zip(*batch)

    if isinstance(imgs[0], Tensor):
        imgs = torch.stack(imgs, 0)

    lengths = [len(cap) for cap in captions]
    max_length = max(lengths)
    padded_captions = torch.full(
        (len(captions), max_length), PAD_TOKEN, dtype=torch.long
    )
    for i, cap in enumerate(captions):
        end = lengths[i]
        padded_captions[i, :end] = torch.tensor(cap[:end], dtype=torch.long)
    lengths = torch.tensor(lengths)

    if is_train:
        return imgs, padded_captions, lengths
    else:
        return imgs, padded_captions, lengths, all_captions


class CaptionDataset(Dataset):
    """Load preprocessed caption dataset."""

    def __init__(
        self,
        root: str | Path,
        preprocess_dir: str | Path, 
        split: DatasetSplit = "train",
        transform=None,
        cache=False,
        max_cap_len=-1,
    ):
        super().__init__()
        self.split = split
        assert self.split in typing.get_args(DatasetSplit)

        root = Path(root)
        preprocess_dir = Path(preprocess_dir) 

        self.root = root
        self.preprocess_dir = preprocess_dir

        if not root.is_dir():
            raise RuntimeError(f"Image root does not exist: {root}")

        self.img_dir = root / "Images"
        if not self.img_dir.is_dir():
            self.img_dir = root / "images"

        # Note on Load, File. Refer to the archived source for the detailed original comment.
        enc_path = preprocess_dir / "captions_encoded.json"
        vocab_path = preprocess_dir / "vocab.json"
        if not enc_path.exists():
            raise FileNotFoundError(f"Missing {enc_path}")
        if not vocab_path.exists():
            raise FileNotFoundError(f"Missing {vocab_path}")

        with open(enc_path, "r") as f:
            captions: list[dict] = json.load(f)
        with open(vocab_path, "r") as f:
            self.vocab: dict[str, int] = json.load(f)

        self.rev_vocab: dict[int, str] = {i: w for w, i in self.vocab.items()}

        # Note on Data. Refer to the archived source for the detailed original comment.
        self.img_names: list[str] = []
        self.captions: list[dict] = []
        max_cap_len_in_data = 0
        for caption in captions:
            if caption["split"] != self.split:
                continue
            img_idx = len(self.img_names)
            self.img_names.append(caption["filename"])
            for sentence in caption["sentences"]:
                max_cap_len_in_data = max(max_cap_len_in_data, len(sentence))
                self.captions.append({"img_idx": img_idx, "caption": sentence})
        self.max_cap_len = max_cap_len_in_data if max_cap_len <= 0 else max_cap_len
        self.cpi = len(captions[0]["sentences"]) if captions else 5
        self.transform = transform

        # Note on Image. Refer to the archived source for the detailed original comment.
        self.img_cache = []
        if cache:
            cache_dir = preprocess_dir / "cache" / self.split
            cache_dir.mkdir(parents=True, exist_ok=True)
            for name in tqdm(self.img_names, desc=f"Caching {self.split} images"):
                img = Image.open(self.img_dir / name).convert("RGB")
                if self.transform:
                    img = self.transform(img)
                self.img_cache.append(img)

    def __getitem__(self, idx: int):
        if self.img_cache:
            img = self.img_cache[self.captions[idx]["img_idx"]]
        else:
            img = Image.open(self.img_dir / self.img_names[self.captions[idx]["img_idx"]]).convert("RGB")
            if self.transform:
                img = self.transform(img)
        caption: list[int] = self.captions[idx]["caption"]

        if self.split == "train":
            return img, caption
        else:
            start_idx = (idx // self.cpi) * self.cpi
            end_idx = start_idx + self.cpi
            all_captions = [cap["caption"] for cap in self.captions[start_idx:end_idx]]
            return img, caption, all_captions

    def __len__(self):
        return len(self.captions)

Writing caption_dataset.py


### Model

This cell documents the image caption generation experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [6]:
!mkdir -p /kaggle/working/models

In [7]:
%%writefile models/encoder.py

import torch
import torchvision.models as models
from torch import Tensor, nn
from torchvision.models import ResNet50_Weights


class Encoder(nn.Module):
    """Encoder model."""

    def __init__(self, encoded_size=(7, 7), finetune=False):
        super().__init__()
        # Note on Training, Load. Refer to the archived source for the detailed original comment.
        resnet = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        # 移除最后两层（平均池化和全连接层）
        self.resnet = nn.Sequential(*list(resnet.children())[:-2])
        # 自适应池化到固定尺寸
        self.pool = nn.AdaptiveAvgPool2d(encoded_size)
        # 默认不微调
        self.finetune(finetune)

    def forward(self, images: Tensor):
        """Extract image features.

        Args:
            images (Tensor): Input images with shape (batch, channels, height, width).

        Returns:
            Tensor: Extracted features with shape (batch, num_pixels, feature_dim).
        """
        # Note on Features. Refer to the archived source for the detailed original comment.
        x: Tensor = self.resnet(images)
        # 池化到固定尺寸
        x: Tensor = self.pool(x)
        # B C H W -> B (H W) C
        x = x.permute(0, 2, 3, 1).flatten(1, 2)
        return x

    def finetune(self, finetune=True):
        # Note on Parameters. Refer to the archived source for the detailed original comment.
        self.resnet.requires_grad_(False)
        # 仅微调后几层
        # for child in list(self.resnet.children())[5:]:
        #     child.requires_grad_(finetune)

        # 只微调最后三个残差块
        for name, child in self.resnet.named_children():
            if name in ['layer2', 'layer3', 'layer4']:
                child.requires_grad_(finetune)

        # 冻结部分保持 eval 模式（固定 BN/Dropout 等）
        if finetune:
            # Note on Training. Refer to the archived source for the detailed original comment.
            for name, child in self.resnet.named_children():
                if name in ['layer2', 'layer3', 'layer4']:
                    child.train()
                else:
                    child.eval()
        else:
            # 完全不微调时，整个 backbone 设为 eval
            self.resnet.eval()

Writing models/encoder.py


In [8]:
%%writefile models/decoder.py

import torch
from torch import Tensor, nn

from .attention import AdditiveAttention


class RNNDecoder(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        hidden_dim: int,
        vocab_size: int,
        feature_dim: int,
        dropout=0.5,
        rnn_cell=nn.LSTMCell,
    ):
        """Initialize RNNDecoder.

        Args:
            embed_dim (int): Embedding dimension.
            hidden_dim (int): Hidden dimension.
            vocab_size (int): Vocabulary size.
            feature_dim (int): Feature dimension.
            dropout (float): Dropout rate. Default: 0.5.
            rnn_cell (Module): RNN cell module. Default: nn.LSTMCell.
        """
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # ============ TODO: Init layers ============
        # Decoder expects these layers:
        # Note on Input. Refer to the archived source for the detailed original comment.
        # - lstm: lstm cell
        # - init_h: linear projection to init hidden state from image features
        #    Note on Image, Features. Refer to the archived source for the detailed original comment.
        # - init_c: linear projection to init cell state from image features
        #    Note on Image, Features. Refer to the archived source for the detailed original comment.
        # - dropout: output dropout layer
        # - fc: output linear projection全连接层
        # raise NotImplementedError
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = rnn_cell(embed_dim, hidden_dim)
        self.lstm_cell = self.lstm
        self.init_h = nn.Linear(feature_dim, hidden_dim)
        self.init_c = nn.Linear(feature_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
        # ===========================================

        # initialize some layers with the uniform distribution
        self.embedding.weight.data.uniform_(-0.1, 0.1)
        self.fc.bias.data.fill_(0)
        self.fc.weight.data.uniform_(-0.1, 0.1)

    def init_hidden_state(self, features: Tensor) -> tuple[Tensor, Tensor]:
        """Initialize hidden states for RNNDecoder.

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).

        Returns:
            tuple: (hidden_state, cell_state) both with shape (batch, hidden_dim).
        """
        # ========== TODO: Init hidden states ==========
        # raise NotImplementedError
        pooled_features = features.mean(dim=1)# 对像素维度取平均，使得 features 变为 (batch, feature_dim)
        h = self.init_h(pooled_features)   # (batch_size, hidden_size)
        c = self.init_c(pooled_features)
        # ==============================================
        return h, c

    def forward(self, features: Tensor, captions: Tensor):
        """Forward pass of RNNDecoder.

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).
            captions (Tensor): Input captions with shape (batch, max_caption_length).

        Returns:
            Tensor: Output predictions with shape (batch, max_caption_length, vocab_size).
        """
        B = features.shape[0]
        T = captions.shape[1]

        # 词嵌入
        embeddings: Tensor = self.embedding(captions)
        # Note on Initialize. Refer to the archived source for the detailed original comment.
        h, c = self.init_hidden_state(features)

        outputs: list[Tensor] = []
        # Decode step by step
        for t in range(T - 1):
            # ======== TODO: implement decode step ========
            # Hint 1: apply dropout just before fc
            # Hint 2: use teacher-forcing here
            # raise NotImplementedError
            current_embed = embeddings[:, t, :] # 取当前时间步的词嵌入 (batch_size, embed_size)
            h, c = self.lstm_cell(current_embed, (h, c))   # 传入 LSTM Cell，更新 h 和 c
            h_dropped = self.dropout(h)   # 经过 Dropout
            
            logits = self.fc(h_dropped)
            outputs.append(logits)   # 堆叠所有时间步的输出，调整维度为 (batch_size, seq_len, vocab_size)
            # =============================================

        # Note on Output. Refer to the archived source for the detailed original comment.
        return torch.stack(outputs, dim=1)

    def generate_caption(
        self, features: Tensor, word_map: dict[str, int], max_len=30, beam_size=1
    ):
        """Generate captions with greedy decoding or beam search (during test time).

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).
            word_map (dict[str, int]): Word map from token to index.
            max_len (int): Maximum caption length. Default: 30.
            beam_size (int): Beam size for beam search. Default: 1 (greedy decoding).

        Returns:
            list[list[int]]: Generated captions.
        """
        B = features.shape[0]
        features = features.view(B, -1, features.shape[-1])
        device = features.device

        # 获取特殊标记
        start_token = word_map["<start>"]
        end_token = word_map["<end>"]

        if beam_size == 1:
            # Note on Use. Refer to the archived source for the detailed original comment.
            # Note on Initialize. Refer to the archived source for the detailed original comment.
            h, c = self.init_hidden_state(features)

            input_tokens = torch.full(
                (B,),
                start_token,
                dtype=torch.long,
                device=device
            )

            captions = [[start_token] for _ in range(B)]
            finished = torch.zeros(B, dtype=torch.bool, device=device)

            # Note on Generate. Refer to the archived source for the detailed original comment.
            for _ in range(max_len - 1):
                # ======== TODO: implement generate step ========
                # Hint 1: apply dropout just before fc
                # Hint 2: a caption ends when it reaches the end token
                # 1. 获取当前 predictions
                # 2. 更新 captions
                # raise NotImplementedError
                current_embed = self.embedding(input_tokens) # 取当前时间步的词嵌入 (batch_size, embed_size)
                h, c = self.lstm_cell(current_embed, (h, c))   # 传入 LSTM Cell，更新 h 和 c  
                logits = self.fc(self.dropout(h))  # 经过 Dropout
                next_tokens = logits.argmax(dim=1)  # 贪心选择概率最高的词

                for i, token in enumerate(next_tokens.tolist()):
                    if not finished[i]:
                        captions[i].append(token)

                finished = finished | (next_tokens == end_token)
                input_tokens = next_tokens

                if finished.all():
                    break
            for i in range(B):
                if captions[i][-1] != end_token:
                    captions[i].append(end_token)
                
                # =============================================
            return captions
            
        else:
            # Note on Use. Refer to the archived source for the detailed original comment.
            captions: list[list[int]] = []
            # Note on Image. Refer to the archived source for the detailed original comment.
            for i in range(B):
                # Note on Image, Features. Refer to the archived source for the detailed original comment.
                img_features = features[i : i + 1]
                # Note on Initialize. Refer to the archived source for the detailed original comment.
                h, c = self.init_hidden_state(img_features)

                # ======== Optional TODO: implement beam search ========
                # Hint: you may need to store beam as (log_prob, hidden_state, cell_state, tokens)
                # 1. Initialize beam with start token
                # 2. Repeat for max_len - 1 steps:
                # 2.1. Compute log probabilities for all possible next tokens
                #      Note: compute only for non-terminated sequences
                # 2.2. Update beam with top-k (k=beam_size) candidates
                # 2.3. Sort beam by log probability, and keep top-k beams
                # 2.4. Stop if all sequences end with end token
                # 3. Select the sequence with highest log probability
                # raise NotImplementedError

                # beam 存储为 (log_prob, hidden, cell, sequence)
                beam = [(0.0, h, c, [])]
                
                for step in range(max_len - 1):
                    new_beam = []
                    # 对每个 beam 进行拓展
                    for log_prob, h_beam, c_beam, seq in beam:
                        # 如果当前序列已经以 <end> 结束，则直接保留
                        if seq and seq[-1] == end_token:
                            new_beam.append((log_prob, h_beam, c_beam, seq))
                            continue

                        # Note on Input. Refer to the archived source for the detailed original comment.
                        current_token = start_token if len(seq) == 0 else seq[-1]
                        current_tensor = torch.tensor([current_token], device=device).long()  # (1,)
                        embed = self.embedding(current_tensor)                                 # (1, embed_dim)
                
                        # 更新LSTM
                        h_new, c_new = self.lstm_cell(embed, (h_beam, c_beam)) 

                        # Note on Output. Refer to the archived source for the detailed original comment.
                        logits = self.fc(self.dropout(h_new))
                        log_probs = torch.log_softmax(logits, dim=1)

                        # 取 top-k（k = beam_size）个候选
                        top_log_probs, top_indices = log_probs.topk(beam_size, dim=1)

                        for k in range(beam_size):
                            cand_log_prob = log_prob + top_log_probs[0, k].item()
                            cand_token = top_indices[0, k].item()
                            new_seq = seq + [cand_token]
                            new_beam.append((cand_log_prob, h_new, c_new, new_seq))
                   
                    # 按 log_prob 降序排序，保留 beam_size 个候选
                    new_beam.sort(key=lambda x: x[0], reverse=True)
                    beam = new_beam[:beam_size]
                
                    # 如果所有 beam 都以 <end> 结尾，提前结束
                    if all(seq and seq[-1] == end_token for _, _, _, seq in beam):
                        break

                # 选择概率最高的序列
                best_seq = beam[0][3]
                full_seq = [start_token] + best_seq # 将 <start> 添加到开头
                
                # 若未以 <end> 结尾，添加（通常 beam 会在结束前停止，但为安全）
                if full_seq[-1] != end_token:
                    full_seq.append(end_token)
                    
                captions.append(full_seq)
                # ======================================================

            return captions


class RNNDecoderWithAttention(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        hidden_dim: int,
        vocab_size: int,
        feature_dim: int,
        dropout=0.5,
        rnn_cell=nn.LSTMCell,
        attn_impl=AdditiveAttention,
    ):
        """Initialize RNNDecoder.

        Args:
            embed_dim (int): Embedding dimension.
            hidden_dim (int): Hidden dimension.
            vocab_size (int): Vocabulary size.
            feature_dim (int): Feature dimension.
            dropout (float): Dropout rate. Default: 0.5.
            rnn_cell (Module): RNN cell module. Default: nn.LSTMCell.
            attn_impl (Module): Attention module. Default: AdditiveAttention.
        """
        super().__init__()
        self.hidden_dim = hidden_dim
        # ============ TODO: Init layers ============
        # Decoder expects these layers:
        # - embedding: input embedding
        # - lstm: lstm cell
        # - attention: attention module
        # - init_h: linear projection to init hidden state from image features
        # - init_c: linear projection to init cell state from image features
        # - dropout: output dropout layer
        # - fc: output linear projection
        # raise NotImplementedError

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = rnn_cell(embed_dim + feature_dim, hidden_dim)
        self.lstm_cell = self.lstm
        self.init_h = nn.Linear(feature_dim, hidden_dim)
        self.init_c = nn.Linear(feature_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

        # Note on Note. Refer to the archived source for the detailed original comment.
        self.attention = AdditiveAttention(feature_dim, hidden_dim)
        
        # ===========================================

        # initialize some layers with the uniform distribution
        self.embedding.weight.data.uniform_(-0.1, 0.1)
        self.fc.bias.data.fill_(0)
        self.fc.weight.data.uniform_(-0.1, 0.1)

    def init_hidden_state(self, features: Tensor) -> tuple[Tensor, Tensor]:
        """Initialize hidden states for RNNDecoder.

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).

        Returns:
            tuple: (hidden_state, cell_state) both with shape (batch, hidden_dim).
        """
        # ========== TODO: Init hidden states ==========
        # Same as RNNDecoder, you can copy the code
        # raise NotImplementedError
        pooled_features = features.mean(dim=1)# 对像素维度取平均，使得 features 变为 (batch, feature_dim)
        h = self.init_h(pooled_features)   # (batch_size, hidden_size)
        c = self.init_c(pooled_features)
        # ==============================================
        return h, c

    def forward(self, features: Tensor, captions: Tensor):
        """Forward pass of RNNDecoder.

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).
            captions (Tensor): Input captions with shape (batch, max_caption_length).

        Returns:
            Tensor: Output predictions with shape (batch, max_caption_length, vocab_size).
        """
        B = features.shape[0]
        T = captions.shape[1]

        # 词嵌入
        embeddings = self.embedding(captions)
        # Note on Initialize. Refer to the archived source for the detailed original comment.
        h, c = self.init_hidden_state(features)

        outputs = []
        # 逐词解码
        for t in range(T - 1):
            # ======== TODO: implement decode step ========
            # Hint 1: apply dropout just before fc
            # Hint 2: use teacher-forcing here
            # raise NotImplementedError
            current_embed = embeddings[:, t, :] # 取当前时间步的词嵌入 (batch_size, embed_size)
            # Note on Image, Features. Refer to the archived source for the detailed original comment.
            context, alpha = self.attention(h, features)

            # 将 context 投影到 embed_dim 后与词嵌入相加
            context, alpha = self.attention(h, features)
            lstm_input = torch.cat([current_embed, context], dim=1)

            h, c = self.lstm_cell(lstm_input, (h, c))
            logits = self.fc(self.dropout(h))
            outputs.append(logits)
            # =============================================

        # Note on Output. Refer to the archived source for the detailed original comment.
        return torch.stack(outputs, dim=1)

    def generate_caption(
        self, features: Tensor, word_map: dict[str, int], max_len=30, beam_size=1
    ):
        """Generate captions with greedy decoding or beam search (during test time).

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).
            word_map (dict[str, int]): Word map from token to index.
            max_len (int): Maximum caption length. Default: 30.
            beam_size (int): Beam size for beam search. Default: 1 (greedy decoding).

        Returns:
            list[list[int]]: Generated captions.
        """
        B = features.shape[0]
        features = features.view(B, -1, features.shape[-1])
        device = features.device

        # 获取特殊标记
        start_token = word_map["<start>"]
        end_token = word_map["<end>"]

        if beam_size == 1:
            # Note on Use. Refer to the archived source for the detailed original comment.
            # Note on Initialize. Refer to the archived source for the detailed original comment.
            h, c = self.init_hidden_state(features)

            input_tokens = torch.full(
                (B,),
                start_token,
                dtype=torch.long,
                device=device
            )

            captions = [[start_token] for _ in range(B)]
            finished = torch.zeros(B, dtype=torch.bool, device=device)

            # Note on Generate. Refer to the archived source for the detailed original comment.
            for _ in range(max_len - 1):
                # ======== TODO: implement generate step ========
                # Hint 1: apply dropout just before fc
                # Hint 2: a caption ends when it reaches the end token
                # 1. 获取当前 predictions
                # 2. 更新 captions
                # raise NotImplementedError
                current_embed = self.embedding(input_tokens) 

                # attention
                context, alpha = self.attention(h, features)
                lstm_input = torch.cat([current_embed, context], dim=1)

                h, c = self.lstm_cell(lstm_input, (h, c))   # 更新 h 和 c  
                logits = self.fc(self.dropout(h))  # 经过 Dropout

                next_tokens = logits.argmax(dim=1)  # 贪心选择概率最高的词

                for i, token in enumerate(next_tokens.tolist()):
                    if not finished[i]:
                        captions[i].append(token)

                finished = finished | (next_tokens == end_token)
                input_tokens = next_tokens

                if finished.all():
                    break
                    
            for i in range(B):
                if captions[i][-1] != end_token:
                    captions[i].append(end_token)
                    
                # =============================================
            return captions

            
        else:
            # Note on Use. Refer to the archived source for the detailed original comment.
            captions: list[list[int]] = []
            # Note on Image. Refer to the archived source for the detailed original comment.
            for i in range(B):
                # Note on Image, Features. Refer to the archived source for the detailed original comment.
                img_features = features[i : i + 1]
                # Note on Initialize. Refer to the archived source for the detailed original comment.
                h, c = self.init_hidden_state(img_features)

                # ======== Optional TODO: implement beam search ========
                # Hint: you may need to store beam as (log_prob, hidden_state, cell_state, tokens)
                # 1. Initialize beam with start token
                # 2. Repeat for max_len - 1 steps:
                # 2.1. Compute log probabilities for all possible next tokens
                #      Note: compute only for non-terminated sequences
                # 2.2. Update beam with top-k (k=beam_size) candidates
                # 2.3. Sort beam by log probability, and keep top-k beams
                # 2.4. Stop if all sequences end with end token
                # 3. Select the sequence with highest log probability
                # raise NotImplementedError

                # beam 存储为 (log_prob, hidden, cell, sequence)
                beam = [(0.0, h, c, [])]
                
                for step in range(max_len - 1):
                    new_beam = []
                    # 对每个 beam 进行拓展
                    for log_prob, h_beam, c_beam, seq in beam:
                        # 如果当前序列已经以 <end> 结束，则直接保留
                        if seq and seq[-1] == end_token:
                            new_beam.append((log_prob, h_beam, c_beam, seq))
                            continue

                        current_token = start_token if len(seq) == 0 else seq[-1]
                        current_tensor = torch.tensor([current_token], device=device).long()
                        embed = self.embedding(current_tensor)

                        # Note on Note. Refer to the archived source for the detailed original comment.
                        context, _ = self.attention(h_beam, img_features)
                        lstm_input = torch.cat([embed, context], dim=1)
                
                        # 更新LSTM
                        h_new, c_new = self.lstm_cell(lstm_input, (h_beam, c_beam)) 

                        # Note on Output. Refer to the archived source for the detailed original comment.
                        logits = self.fc(self.dropout(h_new))

                        log_probs = torch.log_softmax(logits, dim=1)

                        # 取 top-k（k = beam_size）个候选
                        top_log_probs, top_indices = log_probs.topk(beam_size, dim=1)

                        for k in range(beam_size):
                            cand_log_prob = log_prob + top_log_probs[0, k].item()
                            cand_token = top_indices[0, k].item()
                            new_seq = seq + [cand_token]
                            new_beam.append((cand_log_prob, h_new, c_new, new_seq))
        
                    # 排序并保留 top‑beam_size
                    new_beam.sort(key=lambda x: x[0], reverse=True)
                    beam = new_beam[:beam_size]

                    # 如果所有 beam 都以 <end> 结尾，提前结束
                    if all(seq and seq[-1] == end_token for _, _, _, seq in beam):
                        break

                # 选择概率最高的序列
                best_seq = beam[0][3]
                full_seq = [start_token] + best_seq # 将 <start> 添加到开头
                
                # 若未以 <end> 结尾，添加（通常 beam 会在结束前停止，但为安全）
                if full_seq[-1] != end_token:
                    full_seq.append(end_token)
                captions.append(full_seq)
                # ======================================================

            return captions

Writing models/decoder.py


In [9]:
%%writefile models/attention.py

import torch
from torch import Tensor, nn


class AdditiveAttention(nn.Module):
    """Additive attention in Show, Attend, and Tell."""

    def __init__(self, feature_dim: int, hidden_dim: int, attention_dim: int = 512):
        """Initialize AdditiveAttention.

        Args:
            feature_dim (int): Feature dimension.
            hidden_dim (int): Hidden dimension.
        """
        super().__init__()
        # ======== TODO: Init linear layers for attention ========
        # Note on Features. Refer to the archived source for the detailed original comment.
        # Note on Output. Refer to the archived source for the detailed original comment.
        self.Wh = nn.Linear(hidden_dim, attention_dim, bias=False)   # query 变换
        self.Wf = nn.Linear(feature_dim, attention_dim, bias=False)  # key 变换
        self.v = nn.Linear(attention_dim, 1, bias=False)             # 得分向量
        self.tanh = nn.Tanh()
        self.softmax = nn.Softmax(dim=1)  # 在像素维度上归一化
        # ========================================================

    def forward(self, hidden: Tensor, features: Tensor) -> tuple[Tensor, Tensor]:
        """Forward pass of AdditiveAttention.

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).
            hidden (Tensor): Hidden state with shape (batch, hidden_dim).

        Returns:
            Tensor: Attention output with shape (batch, num_pixels, hidden_dim).
        """
        # ============ TODO: forward pass ============
        # e = v * tanh(Wf * features + Wh * hidden)
        # alpha = softmax(e)
        # context = sum(alpha * features)
        # raise NotImplementedError
        # 将 hidden 扩展为 (batch, 1, hidden_dim) 以匹配 features 的序列维度
        hidden_exp = hidden.unsqueeze(1)  # (batch, 1, hidden_dim)

        # Note on Compute, Note. Refer to the archived source for the detailed original comment.
        energy = self.v(
            self.tanh(self.Wf(features) + self.Wh(hidden_exp))
        )  # (batch, num_pixels, 1)

        alpha = self.softmax(energy.squeeze(-1))  # (batch, num_pixels)

        # 上下文向量 = 加权求和
        context = torch.sum(features * alpha.unsqueeze(-1), dim=1)  # (batch, feature_dim)
        
        # ============================================
        return context, alpha

Writing models/attention.py


In [10]:
%%writefile models/__init__.py
from .attention import AdditiveAttention
from .decoder import RNNDecoder, RNNDecoderWithAttention
from .encoder import Encoder

Writing models/__init__.py


### Experiment Notes

This cell documents the image caption generation experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [11]:
%%writefile eval_checkpoints.py

import argparse
import json
from pathlib import Path

import pandas as pd
import torch
import torchvision.transforms as transforms
from nltk.translate.bleu_score import corpus_bleu
from torch.utils.data import DataLoader
from tqdm import tqdm

from caption_dataset import CaptionDataset, collate_fn
from models import Encoder, RNNDecoder, RNNDecoderWithAttention

# Note on Model, Path. Refer to the archived source for the detailed original comment.
ckpt_paths = [
    Path("/kaggle/input/models/fdiiiiii/flickr8k-best-models/pytorch/default/1/Baseline_best_model.pt"), #baseline
    Path("/kaggle/input/models/fdiiiiii/flickr8k-best-models/pytorch/default/1/SR_best_model.pt"), # stronger regulation
    Path("/kaggle/input/models/fdiiiiii/flickr8k-best-models/pytorch/default/1/Att_best_model.pt"), #attention
    Path("/kaggle/input/models/fdiiiiii/flickr8k-best-models/pytorch/default/1/ASS_best_model.pt"), #attetion+scheduled sampling
    Path("/kaggle/input/models/fdiiiiii/flickr8k-best-models/pytorch/default/1/finetune_best_model.pt"), #finetune
]


@torch.no_grad()
def evaluate_model(encoder, decoder, dataloader, vocab, device, beam_size=1, max_len=30):
    encoder.eval()
    decoder.eval()

    hypotheses = []
    references = []

    for imgs, captions, lengths, all_captions in tqdm(
        dataloader, desc=f"Evaluating beam={beam_size}"
    ):
        imgs = imgs.to(device)

        features = encoder(imgs)
        preds = decoder.generate_caption(
            features,
            vocab,
            max_len=max_len,
            beam_size=beam_size,
        )

        # remove <start> and <end>
        cleaned_preds = []
        for pred in preds:
            cleaned_preds.append(pred[1:-1])

        cleaned_refs = []
        for caps in all_captions:
            cleaned_caps = []
            for cap in caps:
                cleaned_caps.append(cap[1:-1])
            cleaned_refs.append(cleaned_caps)

        hypotheses.extend(cleaned_preds)
        references.extend(cleaned_refs)

    bleu4 = corpus_bleu(
        references,
        hypotheses,
        weights=(0.25, 0.25, 0.25, 0.25),
    )
    return bleu4


def load_checkpoint_model(ckpt_path, vocab_size, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    cfg = ckpt["config"]

    finetune_encoder = cfg.get("finetune_encoder", False)
    use_attention = cfg.get("use_attention", False)

    encoder = Encoder(encoded_size=(7, 7), finetune=finetune_encoder).to(device)

    # Probe feature dimension
    with torch.no_grad():
        dummy = torch.zeros(1, 3, 224, 224, device=device)
        feature_dim = encoder(dummy).shape[-1]

    if use_attention:
        decoder = RNNDecoderWithAttention(
            cfg.get("embed_dim", 512),
            cfg.get("hidden_dim", 1024),
            vocab_size,
            feature_dim,
            dropout=cfg.get("dropout", 0.2),
        ).to(device)
    else:
        decoder = RNNDecoder(
            cfg.get("embed_dim", 512),
            cfg.get("hidden_dim", 1024),
            vocab_size,
            feature_dim,
            dropout=cfg.get("dropout", 0.2),
        ).to(device)

    encoder.load_state_dict(ckpt["encoder"])
    decoder.load_state_dict(ckpt["decoder"])

    return encoder, decoder, cfg


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset_dir", type=str, default="/kaggle/input/datasets/adityajn105/flickr8k")
    parser.add_argument("--preprocess_dir", type=str, default="/kaggle/input/datasets/fdiiiiii/flickr8k")
    parser.add_argument("--output_dir", type=str, default="output")
    parser.add_argument("--split", type=str, default="test", choices=["val", "test"])
    parser.add_argument("--beam_sizes", type=int, nargs="+", default=[1, 3, 5])
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--num_workers", type=int, default=4)
    parser.add_argument("--device", type=str, default="cuda")
    parser.add_argument("--max_len", type=int, default=30)
    args = parser.parse_args()

    if args.device == "cuda" and not torch.cuda.is_available():
        args.device = "cpu"

    device = args.device

    transform = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ]
    )

    dataset = CaptionDataset(
        args.dataset_dir,
        preprocess_dir=args.preprocess_dir,
        split=args.split,
        transform=transform,
        cache=False,
    )

    vocab = dataset.vocab
    vocab_size = len(vocab)

    dataloader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        collate_fn=collate_fn,
        pin_memory=True,
    )

    # ckpt_paths = sorted(Path(args.output_dir).glob("*/best_model.pt"))

    if len(ckpt_paths) == 0:
        raise FileNotFoundError(f"No best_model.pt found under {args.output_dir}")

    rows = []

    for ckpt_path in ckpt_paths:
        print("\n" + "=" * 80)
        print(f"Loading checkpoint: {ckpt_path}")

        encoder, decoder, cfg = load_checkpoint_model(ckpt_path, vocab_size, device)

        model_name = ckpt_path.parent.name
        use_attention = cfg.get("use_attention", False)
        use_scheduled_sampling = cfg.get("use_scheduled_sampling", 0)
        finetune_encoder = cfg.get("finetune_encoder", False)
        lr = cfg.get("learning_rate", None)
        dropout = cfg.get("dropout", None)

        for beam_size in args.beam_sizes:
            bleu4 = evaluate_model(
                encoder,
                decoder,
                dataloader,
                vocab,
                device,
                beam_size=beam_size,
                max_len=args.max_len,
            )

            row = {
                "checkpoint": model_name,
                "path": str(ckpt_path),
                "split": args.split,
                "beam_size": beam_size,
                "bleu4": bleu4,
                "use_attention": use_attention,
                "use_scheduled_sampling": use_scheduled_sampling,
                "finetune_encoder": finetune_encoder,
                "learning_rate": lr,
                "dropout": dropout,
            }
            rows.append(row)

            print(
                f"{model_name} | attention={use_attention} | "
                f"scheduled_sampling={use_scheduled_sampling} | "
                f"beam={beam_size} | BLEU-4={bleu4:.4f}"
            )

    df = pd.DataFrame(rows)
    
    save_dir = Path(args.output_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / f"beam_search_comparison_{args.split}.csv"
    df.to_csv(save_path, index=False)

    print("\nFinal results:")
    print(df.sort_values("bleu4", ascending=False))
    print(f"\nSaved to: {save_path}")


if __name__ == "__main__":
    main()

Writing eval_checkpoints.py


In [12]:
# Note on Testing. Refer to the archived source for the detailed original comment.
!python eval_checkpoints.py \
  --output_dir /kaggle/working/output \
  --split test \
  --beam_sizes 1 3 5 7 \
  --batch_size 32 \
  --num_workers 4


Loading checkpoint: /kaggle/input/models/fdiiiiii/flickr8k-best-models/pytorch/default/1/Baseline_best_model.pt
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 165MB/s]
Evaluating beam=1: 100%|██████████████████████| 157/157 [00:18<00:00,  8.62it/s]
1 | attention=False | scheduled_sampling=0 | beam=1 | BLEU-4=0.1963
Evaluating beam=3: 100%|██████████████████████| 157/157 [01:32<00:00,  1.70it/s]
1 | attention=False | scheduled_sampling=0 | beam=3 | BLEU-4=0.2234
Evaluating beam=5: 100%|██████████████████████| 157/157 [02:39<00:00,  1.01s/it]
1 | attention=False | scheduled_sampling=0 | beam=5 | BLEU-4=0.2168
Evaluating beam=7: 100%|██████████████████████| 157/157 [03:54<00:00,  1.50s/it]
1 | attention=False | scheduled_sampling=0 | beam=7 | BLEU-4=0.2129

Loading checkpoint: /kaggle/input/models/fdiiiiii/flickr8k-best-models/pytor

### Experiment / Results / Generate

This cell documents the image caption generation experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.
